# RESUME Training from Checkpoint

**Resume quantum training from epoch 6**  
**Then train classical model from scratch**

This uses the DEBUG configuration (256 batch size, 2000 drugs, 3 layers)

In [ ]:
# Configuration - MUST MATCH ORIGINAL DEBUG RUN
import os
os.environ['OMP_NUM_THREADS'] = '16'
os.environ['MKL_NUM_THREADS'] = '16'

DATA_DIR = "/media/priyanshu/SD/othercode/data"
SAVE_DIR = "./debug_comparison_results"  # SAME AS ORIGINAL
CHECKPOINT_PATH = "./debug_comparison_results/quantum_best.pt"

# MUST MATCH ORIGINAL CONFIG
MAX_DRUGS = 2000
SEED = 42
NUM_QUBITS = 6
NUM_QLAYERS = 3
HIDDEN_DIM = 128
EPOCHS = 100
BATCH_SIZE = 256
LEARNING_RATE_QUANTUM = 0.005
LEARNING_RATE_CLASSICAL = 0.0005
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 15
DEVICE = 'cuda'
QUANTUM_DEVICE = 'lightning.qubit'
NUM_WORKERS = 8
PIN_MEMORY = True
VERBOSE = 2

print(f"📂 RESUME MODE")
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Will continue quantum training, then train classical")

In [ ]:
import importlib
import drug_patient_qgnn.data_processing
import drug_patient_qgnn

importlib.reload(drug_patient_qgnn.data_processing)
importlib.reload(drug_patient_qgnn)

%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import json

from drug_patient_qgnn import (
    DrugPatientDataProcessor,
    QuantumDrugPatientGNN,
    set_seed,
    print_model_summary,
    print_device_info
)

set_seed(SEED)
print_device_info()

In [ ]:
# Load data (SAME AS ORIGINAL)
print(f"Loading data...\n")
start_time = datetime.now()

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)

stats = processor.get_statistics()
print(f"\nData loaded in {(datetime.now() - start_time).total_seconds():.1f}s")

In [ ]:
# Prepare datasets (SAME AS ORIGINAL)
graph = processor.graph
drug_features = graph.get_drug_features_matrix()
patient_features = graph.get_patient_features_matrix()
edge_index, edge_features = graph.get_edge_index()
labels = graph.get_edge_labels()

interaction_data = []
for idx in range(edge_index.shape[1]):
    drug_idx = int(edge_index[0, idx])
    patient_idx = int(edge_index[1, idx])
    
    interaction_data.append({
        'drug_features': drug_features[drug_idx].tolist(),
        'patient_features': patient_features[patient_idx].tolist(),
        'label': float(labels[idx])
    })

df_pandas = pd.DataFrame(interaction_data)

train_pd, val_pd = train_test_split(
    df_pandas,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=df_pandas['label']
)

print(f"Training samples:   {len(train_pd):,}")
print(f"Validation samples: {len(val_pd):,}")

class InteractionDataset(Dataset):
    def __init__(self, df):
        self.drug_features = np.stack(df['drug_features'].values)
        self.patient_features = np.stack(df['patient_features'].values)
        self.labels = df['label'].values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.drug_features[idx], dtype=torch.float32),
            torch.tensor(self.patient_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

train_dataset = InteractionDataset(train_pd)
val_dataset = InteractionDataset(val_pd)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True
)

print(f"\n✓ DataLoaders ready")

In [ ]:
# Training functions (SAME AS DEBUG)
def train_epoch(model, optimizer, criterion, loader, device, show_progress=True):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training', disable=not show_progress)
    
    for batch_idx, (drug_features, patient_features, labels) in enumerate(pbar):
        batch_start = datetime.now()
        
        drug_features = drug_features.to(device, non_blocking=True)
        patient_features = patient_features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        outputs = model(drug_features, patient_features).squeeze(-1)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        batch_time = (datetime.now() - batch_start).total_seconds()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'batch_time': f'{batch_time:.1f}s'})
        
        if VERBOSE >= 2 and (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)}: loss={loss.item():.4f}, time={batch_time:.1f}s")
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}

def evaluate(model, criterion, loader, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for drug_features, patient_features, labels in tqdm(loader, desc='Validation', leave=False):
            drug_features = drug_features.to(device, non_blocking=True)
            patient_features = patient_features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(drug_features, patient_features).squeeze(-1)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }

print("✓ Training functions defined")

## Resume Quantum Training from Checkpoint

In [ ]:
drug_dim = len(drug_features[0])
patient_dim = len(patient_features[0])

print("Creating Quantum Model...")
quantum_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,
    hidden_dim=HIDDEN_DIM,
    use_quantum=True,
    device_name=QUANTUM_DEVICE
)

quantum_model = quantum_model.to(DEVICE)

# Load checkpoint
print(f"\n📂 Loading checkpoint from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH)

quantum_model.load_state_dict(checkpoint['model_state_dict'])
start_epoch = checkpoint['epoch'] + 1
best_val_auc = checkpoint['best_auc']
history = checkpoint['history']

print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"  Best AUC so far: {best_val_auc:.4f}")
print(f"  Resuming from epoch {start_epoch}\n")

# Setup optimizer and scheduler
optimizer = torch.optim.AdamW(quantum_model.parameters(), lr=LEARNING_RATE_QUANTUM, weight_decay=1e-5)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)
criterion = torch.nn.BCEWithLogitsLoss()

print(f"\n{'='*70}")
print(f"RESUMING Quantum Training from Epoch {start_epoch}")
print(f"{'='*70}\n")

patience_counter = 0
start_time = datetime.now()

for epoch in range(start_epoch, EPOCHS):
    epoch_start = datetime.now()
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch+1}/{EPOCHS} - Started at {epoch_start.strftime('%H:%M:%S')}")
    print(f"{'='*70}")
    
    train_metrics = train_epoch(quantum_model, optimizer, criterion, train_loader, DEVICE)
    val_metrics = evaluate(quantum_model, criterion, val_loader, DEVICE)
    
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_metrics['auc'])
    current_lr = optimizer.param_groups[0]['lr']
    
    if current_lr != old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} → {current_lr:.6f}")
    
    # Update history
    history['train_loss'].append(train_metrics['loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])
    history['val_auc'].append(val_metrics['auc'])
    history['val_precision'].append(val_metrics['precision'])
    history['val_recall'].append(val_metrics['recall'])
    history['val_f1'].append(val_metrics['f1'])
    history['learning_rates'].append(current_lr)
    
    epoch_time = (datetime.now() - epoch_start).total_seconds()
    
    print(f"\n{'-'*70}")
    print(f"Epoch {epoch+1} Results ({epoch_time/60:.1f}min):")
    print(f"  Train: loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}")
    print(f"  Val:   loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
          f"auc={val_metrics['auc']:.4f}, f1={val_metrics['f1']:.4f}")
    print(f"  Best AUC so far: {best_val_auc:.4f}")
    print(f"{'-'*70}")
    
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': quantum_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_auc': best_val_auc,
            'history': history
        }, os.path.join(SAVE_DIR, "quantum_best.pt"))
        
        print(f"✓ New best AUC: {best_val_auc:.4f} (saved)")
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
    
    # Save history every epoch
    with open(os.path.join(SAVE_DIR, "quantum_history.json"), 'w') as f:
        json.dump(history, f, indent=2)
    
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

total_time = (datetime.now() - start_time).total_seconds()
print(f"\n✅ Quantum training complete!")
print(f"  Total additional time: {total_time/3600:.2f} hours")
print(f"  Best AUC: {best_val_auc:.4f}")

quantum_history = history
quantum_best_auc = best_val_auc

## Train Classical Model (From Scratch)

In [ ]:
print("\nCreating Classical Model...")
classical_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,
    hidden_dim=HIDDEN_DIM,
    use_quantum=False
)

classical_model = classical_model.to(DEVICE)
optimizer_classical = torch.optim.AdamW(classical_model.parameters(), lr=LEARNING_RATE_CLASSICAL, weight_decay=1e-5)
scheduler_classical = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_classical, mode='max', factor=0.5, patience=5
)

print(f"\n{'='*70}")
print(f"Training CLASSICAL Model")
print(f"{'='*70}\n")

classical_history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_auc': [],
    'val_precision': [], 'val_recall': [], 'val_f1': [],
    'learning_rates': []
}

best_classical_auc = 0.0
patience_counter_classical = 0
start_time_classical = datetime.now()

for epoch in range(EPOCHS):
    epoch_start = datetime.now()
    
    train_metrics = train_epoch(classical_model, optimizer_classical, criterion, train_loader, DEVICE, show_progress=False)
    val_metrics = evaluate(classical_model, criterion, val_loader, DEVICE)
    
    old_lr = optimizer_classical.param_groups[0]['lr']
    scheduler_classical.step(val_metrics['auc'])
    current_lr = optimizer_classical.param_groups[0]['lr']
    
    classical_history['train_loss'].append(train_metrics['loss'])
    classical_history['train_acc'].append(train_metrics['accuracy'])
    classical_history['val_loss'].append(val_metrics['loss'])
    classical_history['val_acc'].append(val_metrics['accuracy'])
    classical_history['val_auc'].append(val_metrics['auc'])
    classical_history['val_precision'].append(val_metrics['precision'])
    classical_history['val_recall'].append(val_metrics['recall'])
    classical_history['val_f1'].append(val_metrics['f1'])
    classical_history['learning_rates'].append(current_lr)
    
    epoch_time = (datetime.now() - epoch_start).total_seconds()
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] "
              f"Epoch {epoch+1:3d}/{EPOCHS} ({epoch_time:.1f}s) - "
              f"val_auc: {val_metrics['auc']:.4f} "
              f"[Best: {best_classical_auc:.4f}]")
    
    if val_metrics['auc'] > best_classical_auc:
        best_classical_auc = val_metrics['auc']
        patience_counter_classical = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': classical_model.state_dict(),
            'optimizer_state_dict': optimizer_classical.state_dict(),
            'best_auc': best_classical_auc,
            'history': classical_history
        }, os.path.join(SAVE_DIR, "classical_best.pt"))
        print(f"  ✓ New best AUC: {best_classical_auc:.4f}")
    else:
        patience_counter_classical += 1
    
    if (epoch + 1) % 10 == 0:
        with open(os.path.join(SAVE_DIR, "classical_history.json"), 'w') as f:
            json.dump(classical_history, f, indent=2)
    
    if patience_counter_classical >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

total_time_classical = (datetime.now() - start_time_classical).total_seconds()
print(f"\n✅ Classical training complete!")
print(f"  Total time: {total_time_classical/60:.1f} minutes")
print(f"  Best AUC: {best_classical_auc:.4f}")

classical_best_auc = best_classical_auc

## Final Comparison

In [ ]:
print("\n" + "="*80)
print("🏆 FINAL RESULTS: QUANTUM vs CLASSICAL GNN")
print("="*80)

print(f"\n{'Metric':<25} {'Quantum':<15} {'Classical':<15} {'Difference':<15} {'Winner'}")
print("-"*80)

metrics = [
    ('Best Validation AUC', quantum_best_auc, classical_best_auc),
    ('Final Val Accuracy', quantum_history['val_acc'][-1], classical_history['val_acc'][-1]),
    ('Final Val F1 Score', quantum_history['val_f1'][-1], classical_history['val_f1'][-1]),
]

quantum_wins = 0
for metric_name, quantum_val, classical_val in metrics:
    diff = quantum_val - classical_val
    diff_pct = (diff / classical_val) * 100
    winner = '🏆 QUANTUM' if quantum_val > classical_val else '🏆 Classical'
    if quantum_val > classical_val:
        quantum_wins += 1
    print(f"{metric_name:<25} {quantum_val:<15.4f} {classical_val:<15.4f} {diff:+.4f} ({diff_pct:+.1f}%)  {winner}")

print("\n" + "="*80)
print(f"Quantum wins: {quantum_wins}/3 metrics")
print("="*80)

print(f"\n✅ ALL TRAINING COMPLETE at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")